In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

BASE = Path.cwd().parent
DATA = BASE / "Datasets" / "processed" / "dataset.csv"
RESULTS = BASE / "Results"

df = pd.read_csv(DATA).fillna({"title": "", "abstract": "", "ref_dois": ""})
label_cols = [c for c in df.columns if c.startswith("lbl_")]
names = [c[4:] for c in label_cols]
Y = df[label_cols].values


dev_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20,
                                     random_state=42, shuffle=True)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.25,
                                      random_state=42, shuffle=True)
fit_idx = np.concatenate([train_idx, val_idx])


vocabulary, row_idx, col_idx = {}, [], []
for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        row_idx.append(i); col_idx.append(j)
X_cite = normalize(csr_matrix((np.ones(len(row_idx)), (row_idx, col_idx)),
                              shape=(len(df), len(vocabulary))))


vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2,
                      sublinear_tf=True, stop_words="english")
X_fit = vec.fit_transform((df.title + " " + df.abstract).values[fit_idx])
X_test = vec.transform((df.title + " " + df.abstract).values[test_idx])
text_model = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
text_model.fit(X_fit, Y[fit_idx])


scores_text = text_model.decision_function(X_test)
pred_text = (scores_text >= 0.0).astype(int)
empty = pred_text.sum(1) == 0
pred_text[empty, scores_text[empty].argmax(1)] = 1


S = (X_cite[test_idx] @ X_cite[fit_idx].T).toarray()
k = 20
nn = np.argpartition(-S, k, axis=1)[:, :k]
scores_cite = np.zeros((len(test_idx), Y.shape[1]))
for i in range(len(test_idx)):
    w = S[i, nn[i]]
    if w.sum() > 0:
        scores_cite[i] = (Y[fit_idx][nn[i]] * w[:, None]).sum(0) / w.sum()
pred_cite = (scores_cite >= 0.35).astype(int)
empty = pred_cite.sum(1) == 0
pred_cite[empty, scores_cite[empty].argmax(1)] = 1

Y_test = Y[test_idx]
print(f"test set: {len(test_idx)} papers")
print(f"text     micro-F1 {f1_score(Y_test, pred_text, average='micro'):.3f}")
print(f"citation micro-F1 {f1_score(Y_test, pred_cite, average='micro'):.3f}")

test set: 1639 papers
text     micro-F1 0.750
citation micro-F1 0.752


In [2]:


def confusion_pairs(y_true, y_pred):
    """Count (true field missed, field wrongly predicted) pairs."""
    M = np.zeros((len(names), len(names)))
    for i in range(len(y_true)):
        missed = np.where((y_true[i] == 1) & (y_pred[i] == 0))[0]
        added  = np.where((y_true[i] == 0) & (y_pred[i] == 1))[0]
        for a in missed:
            for b in added:
                M[a, b] += 1
    return M

conf_text = confusion_pairs(Y_test, pred_text)
conf_cite = confusion_pairs(Y_test, pred_cite)

print("TEXT MODEL — most common confusions")
print("(true field was missed, this field was predicted instead)\n")
pairs = [(conf_text[a, b], names[a], names[b])
         for a in range(len(names)) for b in range(len(names)) if a != b]
for count, true_f, pred_f in sorted(pairs, reverse=True)[:12]:
    print(f"  {int(count):>4}  {true_f:<18} -> {pred_f}")

def confusion_pairs(y_true, y_pred):
    
    M = np.zeros((len(names), len(names)))
    for i in range(len(y_true)):
        missed = np.where((y_true[i] == 1) & (y_pred[i] == 0))[0]
        added  = np.where((y_true[i] == 0) & (y_pred[i] == 1))[0]
        for a in missed:
            for b in added:
                M[a, b] += 1
    return M

conf_text = confusion_pairs(Y_test, pred_text)
conf_cite = confusion_pairs(Y_test, pred_cite)

print("TEXT MODEL — most common confusions")
print("(true field was missed, this field was predicted instead)\n")
pairs = [(conf_text[a, b], names[a], names[b])
         for a in range(len(names)) for b in range(len(names)) if a != b]
for count, true_f, pred_f in sorted(pairs, reverse=True)[:12]:
    print(f"  {int(count):>4}  {true_f:<18} -> {pred_f}")

TEXT MODEL — most common confusions
(true field was missed, this field was predicted instead)

    32  CS_InfoSystems     -> CS_AI
    17  CS_AI              -> Engineering_EE
    16  Engineering_EE     -> CS_AI
    16  CS_InfoSystems     -> Engineering_EE
    16  CS_AI              -> CS_InfoSystems
    14  Environmental      -> MaterialsScience
    13  Mathematics        -> CS_AI
    13  Environmental      -> Psychology
    13  CS_InfoSystems     -> Psychology
    12  Chemistry          -> Biotechnology
    12  CS_AI              -> Mathematics
    11  Mathematics        -> CS_InfoSystems
TEXT MODEL — most common confusions
(true field was missed, this field was predicted instead)

    32  CS_InfoSystems     -> CS_AI
    17  CS_AI              -> Engineering_EE
    16  Engineering_EE     -> CS_AI
    16  CS_InfoSystems     -> Engineering_EE
    16  CS_AI              -> CS_InfoSystems
    14  Environmental      -> MaterialsScience
    13  Mathematics        -> CS_AI
    13  Environme

In [3]:


CLUSTER = ["Chemistry", "MaterialsScience", "Physics_Applied"]
cluster_idx = [names.index(n) for n in CLUSTER]

def cluster_share(M):
    
    total = M.sum() - np.trace(M)
    inside = sum(M[a, b] for a in cluster_idx for b in cluster_idx if a != b)
    return inside, total, 100 * inside / total if total else 0

for label, M in [("text", conf_text), ("citation", conf_cite)]:
    inside, total, pct = cluster_share(M)
    print(f"{label:<10} {int(inside):>5} of {int(total):>5} confusions "
          f"are within the Chemistry/Materials/Physics cluster ({pct:.1f}%)")

print("\nThe cluster is 3 of 10 fields. If confusions were spread evenly you")
print("would expect roughly 6-7% of them to fall inside it by chance.")

text          23 of   404 confusions are within the Chemistry/Materials/Physics cluster (5.7%)
citation       8 of   461 confusions are within the Chemistry/Materials/Physics cluster (1.7%)

The cluster is 3 of 10 fields. If confusions were spread evenly you
would expect roughly 6-7% of them to fall inside it by chance.


In [4]:


def exact_match(y_true, y_pred):
    return (y_true == y_pred).all(axis=1)

t_ok = exact_match(Y_test, pred_text)
c_ok = exact_match(Y_test, pred_cite)

groups = {
    "both correct":            t_ok & c_ok,
    "both wrong":             ~t_ok & ~c_ok,
    "text right, cite wrong":  t_ok & ~c_ok,
    "CITE right, text wrong": ~t_ok & c_ok,
}

print(f"{'group':<26}{'papers':>8}{'%':>8}")
for g, mask in groups.items():
    print(f"{g:<26}{mask.sum():>8}{100 * mask.mean():>8.1f}")

rescued = groups["CITE right, text wrong"]
print(f"\n{rescued.sum()} papers were classified correctly by citations")
print("but not by text. These are what the hybrid model gains.")

group                       papers       %
both correct                   717    43.7
both wrong                     500    30.5
text right, cite wrong         238    14.5
CITE right, text wrong         184    11.2

184 papers were classified correctly by citations
but not by text. These are what the hybrid model gains.


In [5]:


test_df = df.iloc[test_idx].copy()
test_df["abstract_words"] = test_df.abstract.apply(lambda t: len(str(t).split()))
test_df["rescued"] = rescued

print(f"{'property':<24}{'rescued':>12}{'others':>12}")
for col, label in [("abstract_words", "abstract length"),
                   ("n_refs", "references"),
                   ("n_labels", "number of labels")]:
    a = test_df[test_df.rescued][col].median()
    b = test_df[~test_df.rescued][col].median()
    print(f"{label:<24}{a:>12.0f}{b:>12.0f}")

print("\nFields of the rescued papers:")
for n in names:
    c = int(test_df[test_df.rescued][f"lbl_{n}"].sum())
    if c:
        print(f"  {c:>3}  {n}")

property                     rescued      others
abstract length                  190         190
references                        58          58
number of labels                   2           1

Fields of the rescued papers:
    8  Biotechnology
   69  Chemistry
   56  CS_AI
   24  CS_InfoSystems
   69  Engineering_EE
    8  Environmental
   61  MaterialsScience
   14  Mathematics
   56  Physics_Applied
    6  Psychology


In [6]:


pd.set_option("display.max_colwidth", 110)
sample = test_df[test_df.rescued].head(8)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    print(f"{i}. {row.title[:105]}")
    print(f"   true: {row.labels}")
    print(f"   abstract: {row.abstract_words} words | refs: {row.n_refs}")
    print()

1. Cyanazine herbicide monitoring as a hazardous substance by a DNA nanostructure biosensor
   true: Environmental
   abstract: 235 words | refs: 35

2. Photothermal Nanozyme-Based Microneedle Patch against Refractory Bacterial Biofilm Infection via Iron-Act
   true: Chemistry|MaterialsScience|Physics_Applied
   abstract: 198 words | refs: 54

3. Software update: The ORCA program system-Version 5.0
   true: Chemistry
   abstract: 83 words | refs: 145

4. Machine Learning for Materials Scientists: An Introductory Guide toward Best Practices
   true: Chemistry|MaterialsScience
   abstract: 113 words | refs: 122

5. Single-Junction Organic Photovoltaic Cell with 19% Efficiency
   true: Chemistry|MaterialsScience|Physics_Applied
   abstract: 154 words | refs: 50

6. A Logic-Based Diagnostic and Therapeutic Hydrogel with Multistimuli Responsiveness to Orchestrate Diabeti
   true: Chemistry|MaterialsScience|Physics_Applied
   abstract: 167 words | refs: 68

7. Deep Learning for Multigrade Br

In [7]:



pairs_df = pd.DataFrame(
    [(names[a], names[b], int(conf_text[a, b]), int(conf_cite[a, b]))
     for a in range(len(names)) for b in range(len(names)) if a != b],
    columns=["true_field", "predicted_instead", "text_count", "citation_count"]
).sort_values("text_count", ascending=False)
pairs_df.to_csv(RESULTS / "confusion_pairs.csv", index=False)


pd.DataFrame([{"group": g, "papers": int(m.sum()), "percent": round(100 * m.mean(), 1)}
              for g, m in groups.items()]).to_csv(
    RESULTS / "model_disagreement.csv", index=False)


test_df[test_df.rescued][
    ["ut", "title", "labels", "abstract_words", "n_refs", "n_labels"]
].to_csv(RESULTS / "rescued_papers.csv", index=False)

print("saved: confusion_pairs.csv, model_disagreement.csv, rescued_papers.csv")

saved: confusion_pairs.csv, model_disagreement.csv, rescued_papers.csv


In [8]:

abs_len = df.abstract.apply(lambda t: len(str(t).split()))
test_len = abs_len.iloc[test_idx].values

len_groups = {
    "no abstract":        test_len == 0,
    "very short (1-99)":  (test_len > 0) & (test_len < 100),
    "short (100-149)":    (test_len >= 100) & (test_len < 150),
    "typical (150-249)":  (test_len >= 150) & (test_len < 250),
    "long (250+)":        test_len >= 250,
}

for g, mask in len_groups.items():
    if mask.sum() < 20:
        print(f"{g:<22} only {mask.sum()} papers - too few")
        continue
    t = f1_score(Y[test_idx][mask], pred_text[mask], average="micro", zero_division=0)
    c = f1_score(Y[test_idx][mask], pred_cite[mask], average="micro", zero_division=0)
    print(f"{g:<22} n={mask.sum():>4}   text {t:.3f}   citation {c:.3f}")

no abstract            only 6 papers - too few
very short (1-99)      n=  73   text 0.823   citation 0.757
short (100-149)        n= 262   text 0.737   citation 0.755
typical (150-249)      n=1064   text 0.752   citation 0.763
long (250+)            n= 234   text 0.733   citation 0.687


In [9]:


print(f"{'group':<26}{'papers':>8}{'median labels':>15}{'% multi-label':>15}")
for g, mask in groups.items():
    sub = test_df[mask]
    print(f"{g:<26}{mask.sum():>8}{sub.n_labels.median():>15.0f}"
          f"{100 * (sub.n_labels > 1).mean():>14.1f}%")

print("\nExact-match rate by number of true labels:")
print(f"{'labels':<8}{'papers':>8}{'text':>8}{'citation':>10}")
for n in [1, 2, 3]:
    m = (test_df.n_labels == n).values
    print(f"{n:<8}{m.sum():>8}{t_ok[m].mean():>8.3f}{c_ok[m].mean():>10.3f}")


pd.DataFrame([{"labels": n,
               "papers": int((test_df.n_labels == n).sum()),
               "text_exact": round(t_ok[(test_df.n_labels == n).values].mean(), 3),
               "citation_exact": round(c_ok[(test_df.n_labels == n).values].mean(), 3)}
              for n in [1, 2, 3]]).to_csv(RESULTS / "exact_match_by_labels.csv", index=False)
print("\nsaved exact_match_by_labels.csv")

group                       papers  median labels  % multi-label
both correct                   717              1          22.5%
both wrong                     500              1          36.6%
text right, cite wrong         238              1          15.5%
CITE right, text wrong         184              2          67.4%

Exact-match rate by number of true labels:
labels    papers    text  citation
1           1134   0.668     0.543
2            319   0.342     0.439
3            186   0.478     0.780

saved exact_match_by_labels.csv


In [10]:

def zscale(a):
    return (a - a.mean()) / (a.std() + 1e-9)

combined = 0.65 * zscale(scores_text) + 0.35 * zscale(scores_cite)
pred_hyb = (combined >= 1.0).astype(int)
empty = pred_hyb.sum(1) == 0
pred_hyb[empty, combined[empty].argmax(1)] = 1

per = pd.DataFrame({
    "field": names,
    "test_papers": Y_test.sum(axis=0),
    "text": f1_score(Y_test, pred_text, average=None, zero_division=0).round(3),
    "citation": f1_score(Y_test, pred_cite, average=None, zero_division=0).round(3),
    "hybrid": f1_score(Y_test, pred_hyb, average=None, zero_division=0).round(3),
})
print(per.to_string(index=False))
print(f"\nhybrid micro-F1 check: {f1_score(Y_test, pred_hyb, average='micro'):.3f}")
per.to_csv(RESULTS / "per_field_f1.csv", index=False)

           field  test_papers  text  citation  hybrid
   Biotechnology          205 0.861     0.806   0.876
       Chemistry          340 0.817     0.821   0.831
           CS_AI          224 0.662     0.713   0.731
  CS_InfoSystems          196 0.575     0.646   0.632
  Engineering_EE          291 0.684     0.692   0.730
   Environmental          193 0.789     0.738   0.799
MaterialsScience          283 0.786     0.786   0.805
     Mathematics          183 0.784     0.755   0.805
 Physics_Applied          212 0.622     0.668   0.679
      Psychology          203 0.874     0.878   0.889

hybrid micro-F1 check: 0.779
